In [1]:
from os import path
from glob import glob
from tqdm import tqdm
import uuid
import csv
import tables
import logging
import pandas as pd
import numpy as np
from os.path import join
from ctapipe.instrument import CameraGeometry
from ctapipe.image import tailcuts_clean, dilate
from astropy import units
from joblib import load

Dicionarios con parámetros de telescopios

In [2]:
# CSV events data
_event_fieldnames = [
    'event_unique_id',  # hdf5 event identifier
    'event_id',  # Unique event identifier
    'source',  # hfd5 filename
    'folder',  # Container hdf5 folder
    'core_x',  # Ground x coordinate
    'core_y',  # Ground y coordinate
    'h_first_int',  # Height firts impact
    'alt',  # Altitute
    'az',  # Azimut
    'mc_energy'  # Monte Carlo Energy
]

# CSV Telescope events data
_telescope_fieldnames = [
    'telescope_id',  # Unique telescope identifier
    'event_unique_id',  # hdf5 event identifier
    'type',  # Telescope type
    'x',  # x array coordinate
    'y',  # y array coordinate
    'z',  # z array coordinate
    'observation_indice'  # Observation indice in table
]
__all__ = [
    'extract_data',
    'generate_dataset', 'load_dataset', 'save_dataset', 'split_dataset',
    'filter_dataset', 'aggregate_dataset', 'describe_dataset', 'load_hillas_dataset',
    'aggregate_hillas_dataset'
]

# Table names and atributes
_events_table = {
    "ML1": "Event_Info",
    "ML2": "Events",
    "DL1": "event"
}

_event_attributes = {
    "ML1": {
        "event_id": "event_number",
        "core_x": "core_x",
        "core_y": "core_y",
        "alt": "alt",
        "az": "az",
        "h_first_int": "h_first_int",
        "mc_energy": "mc_energy",
    },
    "ML2": {
        "event_id": "event_id",
        "core_x": "core_x",
        "core_y": "core_y",
        "alt": "alt",
        "az": "az",
        "h_first_int": "h_first_int",
        "mc_energy": "mc_energy",
    },
    "DL1": {
        "event_id": "event_id",
        "core_x": "true_core_x",
        "core_y": "true_core_y",
        "alt": "true_alt",
        "az": "true_az",
        "h_first_int": "true_h_first_int",
        "mc_energy": "true_energy",
    }
}

_array_info_table = {
    "ML1": "Array_Info",
    "ML2": "Array_Information"
}

_array_attributes = {
    "ML1": {
        "type": "tel_type",
        "telescope_id": "tel_id",
        "x": "tel_x",
        "y": "tel_y",
        "z": "tel_z",
    },
    "ML2": {
        "type": "type",
        "telescope_id": "id",
        "x": "x",
        "y": "y",
        "z": "z",
    },
    "DL1": {
        "type": "type",
        "telescope_id": "tel_id",
        "x": "pos_x",
        "y": "pos_y",
        "z": "pos_z",
    }
}

TELESCOPES = ["LST_LSTCam", "MST_FlashCam", "SST1M_DigiCam"]
TELESCOPES_ALIAS = {
    "ML1": {
        "LST_LSTCam":    "LST", 
        "MST_FlashCam":  "MSTF", 
        "SST1M_DigiCam": "SST1"
    },
    "ML2": {
        "LST_LSTCam":    "LST_LSTCam", 
        "MST_FlashCam":  "MST_FlashCam",
        "SST1M_DigiCam": "SST1M_DigiCam"
    },
    "DL1": {
        "LST_LSTCam":    "LST",
        #"MST_NectarCam": "MST",    #Falta este tipo de cámara. Arreglar después
        "MST_FlashCam":  "MST",
        "SST1M_DigiCam": "SST"
    }
}

_images_attributes = {
    "ML1": {
        "charge":   "image_charge",
        "peakpos":   "image_peak_times",
    },
    "ML2": {
        "charge":   "charge",
        "peakpos":   "peakpos",
    },
    "DL1": {
        "charge":   "image",
        "peakpos":  "peak_time",
    }
}

Función de extract data de data/dataset.py se usa para extraer info del h5 a csv

In [3]:
def extract_data(hdf5_filepath, n_file, version='ML2'):
    """Extract data from one hdf5 file."""

    hdf5_file = tables.open_file(hdf5_filepath, "r")
    source = path.basename(hdf5_filepath)
    folder = path.dirname(hdf5_filepath)

    events_data = []
    telescopes_data = []

    # Array data
    array_data = {}
    # Telescopes Ids
    real_telescopes_id = {}
    ## 'activated_telescopes' are not the real id for each telescopes. These activated_telecopes
    ## are indices related to the event table, but not with  the array information table (telescope info).
    ## In the array info table, all telescope (from different types) are indexed together starting from 1.
    ## But they are in orden, grouped by type (lst, mst and then sst). In the other hand, the Event table
    ## has 3 indices starting from 0, for each telescope type. 
    ## 'real_telescopes_id' translate events indices ('activation_telescope_id') to array ids ('telescope_id').

    if version == "DL1":
        telescope_list = hdf5_file.root.configuration.instrument.subarray.layout
#        for telescope in hdf5_file.root.configuration.instrument.subarray.layout:
#            telescope_type = telescope[_array_attributes[version]["type"]]
#            telescope_type = telescope_type.decode("utf-8") if isinstance(telescope_type, bytes) else telescope_type
#            telescope_id = telescope[_array_attributes[version]["telescope_id"]]

#            if telescope_type not in array_data:
#                array_data[telescope_type] = {}
#                real_telescopes_id[telescope_type] = []

#            array_data[telescope_type][telescope_id] = {
#                "id": telescope_id,
#                "x": telescope[_array_attributes[version]["x"]],
#                "y": telescope[_array_attributes[version]["y"]],
#                "z": telescope[_array_attributes[version]["z"]],
#            }
#            real_telescopes_id[telescope_type].append(telescope_id)

    else:
        telescope_list = hdf5_file.root[_array_info_table[version]]

    for telescope in telescope_list:
        telescope_type = telescope[_array_attributes[version]["type"]]
        telescope_type = telescope_type.decode("utf-8") if isinstance(telescope_type, bytes) else telescope_type
        telescope_id = telescope[_array_attributes[version]["telescope_id"]]
        # HERE
        if telescope_type not in array_data:
            array_data[telescope_type] = {}
            real_telescopes_id[telescope_type] = []

        array_data[telescope_type][telescope_id] = {
            "id": telescope_id,
            "x": telescope[_array_attributes[version]["x"]],
            "y": telescope[_array_attributes[version]["y"]],
            "z": telescope[_array_attributes[version]["z"]],
        }
        real_telescopes_id[telescope_type].append(telescope_id)

#    event_asociation_ids = {} #estructura que guarda la relación entre las id de telescopio, observación y evento

#    if version == "DL1":
#        for tel in hdf5_file.root.dl1.event.telescope.parameters:
#            for row in tel:
#                if row["obs_id"] not in event_asociation_ids:
#                    event_asociation_ids[row["obs_id"]] = {}
#                if row["event_id"] not in event_asociation_ids[row["obs_id"]]:
#                    event_asociation_ids[row["obs_id"]][row["event_id"]] = []
                
#                event_asociation_ids[row["obs_id"]][row["event_id"]].append(row["tel_id"])

    # add uuid to avoid duplicated event numbers 
    try:
        if version == "DL1":
            event_list = hdf5_file.root.simulation.event.subarray.shower
        else:
            event_list = hdf5_file.root[_events_table[version]]
        for i, event in enumerate(event_list):
            #pdb.set_trace()
            # Event data
            if version == "DL1":
                event_unique_id = str(event["obs_id"]) + "_" + str(event["event_id"]) + "_" + str(n_file)
            else:
                event_unique_id = uuid.uuid4().hex[:20]
            event_data = dict(
                event_unique_id=event_unique_id,
                event_id=event[_event_attributes[version]["event_id"]],
                source=source,
                folder=folder,
                core_x=event[_event_attributes[version]["core_x"]],
                core_y=event[_event_attributes[version]["core_y"]],
                h_first_int=event[_event_attributes[version]["h_first_int"]],
                alt=event[_event_attributes[version]["alt"]],
                az=event[_event_attributes[version]["az"]],
                mc_energy=event[_event_attributes[version]["mc_energy"]]
            )
            events_data.append(event_data)

            # Observations data
            ## For each telescope type
#            if version == "DL1":
#                for fila in event_asociation_ids:
#                    #pdb.set_trace()
#                    for columna in event_asociation_ids[fila]:
#                        tel_ids = event_asociation_ids[fila][columna]
#                        for tel_id in tel_ids:
#                            telescope_type = None
#                            for t_type, tels in array_data.items():
#                                if tel_id in tels.keys():
#                                    telescope_type = t_type
#                                    break
#                            if telescope_type is None:
#                                continue
#                            telescope_data = dict(
#                                telescope_id=tel_id,
#                                event_unique_id=event_unique_id,
#                                type=telescope_type,
#                                x=array_data[telescope_type][tel_id]["x"],
#                                y=array_data[telescope_type][tel_id]["y"],
#                                z=array_data[telescope_type][tel_id]["z"],
#                                observation_indice=fila#tentativo hasta averiguar que es el observation indice
#                            )
#                            telescopes_data.append(telescope_data)
            if version != "DL1":
                for telescope_type in TELESCOPES:#Para un evento dado (loop anterior). Se examinan todos los tipos de telescopios.
                    telescope_type_alias = TELESCOPES_ALIAS[version][telescope_type]
                    telescope_indices = f"{telescope_type_alias}_indices"
                    telescopes = event[telescope_indices]#Formato de telescopes? Telescopios de 1 tipo para un evento puntual dado el loop anterior?
                    # number of activated telescopes
                    if version == "ML2":
                        telescope_multiplicity = f"{telescope_type_alias}_multiplicity"
                        multiplicity = event[telescope_multiplicity]
                    else:
                        multiplicity = np.sum(telescopes != 0)#Numero de telescopios de ese tipo activados para un evento dado

                    if multiplicity == 0:  # No telescope of this type were activated
                        continue

                    # Select activated telescopes
                    activation_mask = telescopes != 0
                    activated_telescopes = np.arange(len(telescopes))[activation_mask]#numero en la lista de un tipo de telescopio de telescopios activados
                    observation_indices = telescopes[activation_mask]#los telescopios de cierrto tipo de un evento que son activados?

                    ## For each activated telescope
                    for activate_telescope, observation_indice in zip(activated_telescopes, observation_indices):#cada telescopio de un tipo activado en un evento con su indice
                        # Telescope Data
                        real_telescope_id = real_telescopes_id[telescope_type_alias][activate_telescope]
                        #entonces en este loop se guardan los datos de cada telescopio activado para un evento de un tipo dado
                        telescope_data = dict(
                            telescope_id=real_telescope_id,
                            event_unique_id=event_unique_id,
                            type=telescope_type,
                            x=array_data[telescope_type_alias][real_telescope_id]["x"],
                            y=array_data[telescope_type_alias][real_telescope_id]["y"],
                            z=array_data[telescope_type_alias][real_telescope_id]["z"],
                            observation_indice=observation_indice
                        )
                        telescopes_data.append(telescope_data)
        
#        if version == "DL1":
#            for fila in event_asociation_ids:#obs_id
#               #pdb.set_trace()
#                for columna in event_asociation_ids[fila]:#event_id
#                    tel_ids = event_asociation_ids[fila][columna]
#                    for tel_id in tel_ids:
#                        telescope_type = None
#                        for t_type, tels in array_data.items():
#                            if tel_id in tels.keys():
#                                telescope_type = t_type
#                                break
#                        if telescope_type is None:
#                            continue
#                        telescope_data = dict(
#                            telescope_id=tel_id,
#                            event_unique_id=event_unique_id,
#                            type=telescope_type,
#                            x=array_data[telescope_type][tel_id]["x"],
#                            y=array_data[telescope_type][tel_id]["y"],
#                            z=array_data[telescope_type][tel_id]["z"],
#                            observation_indice=fila#tentativo hasta averiguar que es el observation indice
#                        )
#                        telescopes_data.append(telescope_data)
        
        if version == "DL1":
            telescope_ids_list = hdf5_file.root.configuration.instrument.subarray.layout.read()[:]["tel_id"]
            for event in hdf5_file.root.dl1.event.subarray.trigger:
                tel_ids = telescope_ids_list[event["tels_with_trigger"]]
                for tel_id in tel_ids:
                    telescope_type = None
                    for t_type, tels in array_data.items():
                        if tel_id in tels.keys():
                            telescope_type = t_type
                            break
                    if telescope_type is None:
                        continue
                    telescope_data = dict(
                        telescope_id=tel_id,
                        event_unique_id=str(event["obs_id"]) + "_" + str(event["event_id"]) + "_" + str(n_file),
                        type=telescope_type,
                        x=array_data[telescope_type][tel_id]["x"],
                        y=array_data[telescope_type][tel_id]["y"],
                        z=array_data[telescope_type][tel_id]["z"],
                        observation_indice=str(event["obs_id"]) + "_" + str(event["event_id"]) + "_" + str(tel_id) + "_" + str(n_file)#esta combinación garantiza identificador unico en todos los campos
                    )
                    telescopes_data.append(telescope_data)
    except KeyboardInterrupt:
        logging.info("Extraction stopped.")
    except Exception as err:
        logging.error(err)
        logging.info("Extraction ended by an error.")
    else:
        logging.info("Extraction ended successfully.")
    finally:
        logging.debug(f"Total events: {len(events_data)}")
        logging.debug(f"Total observations: {len(telescopes_data)}")

    return events_data, telescopes_data

Funciones de datos/dataset.py para el procesamiento de los csv y de data/utils.py con para preparar la info entregada por el json de entrenamiento.

In [ ]:
def load_dataset(events_path, telescopes_path, replace_folder=None):
    """Load events.csv and telescopes.csv files into dataframes.
    
    Parameters
    ----------
    events_path : `str`
        Path to events.csv file.
    telescopes_path : `str`
        Path to telescopes.csv file.
    replace_folder : `str` or `None`
        Path to folder containing hdf5 files. Replace the folder 
        column from csv file. Usefull if the csv files are shared
        between different machines. Default None, means no change
        applied.

    Returns
    -------
    dataset : `pd.DataFrame`
        Dataset of observations for reference telescope images.
    """
    # Load data
    events_data = pd.read_csv(events_path, delimiter=";")
    telescopes_data = pd.read_csv(telescopes_path, delimiter=";")

    # Change dataset folder
    if replace_folder is not None:
        events_data.folder = replace_folder

    # Join tables
    dataset = pd.merge(events_data, telescopes_data, on="event_unique_id", validate="1:m")

    return dataset

def aggregate_dataset(dataset, version, az=True, log10_mc_energy=True, hdf5_file=True):
    """
    Perform simple aggegation to targe columns.

    Parameters
    ==========
    az : `bool`, optional
        Translate domain from [0, 2\pi] to [-\pi, \pi]. (default=False)
    log10_mc_energy : `bool`, optional
        Add new log10_mc_energy column, with the logarithm values of mc_energy.
    Returns
    =======
    `pd.DataFrame`
        Dataset with aggregate information.
    """
    #if version == "DL1":
    #    dataset["alt"] = dataset["alt"].apply(lambda deg: np.deg2rad(deg))
    #    dataset["az"] = dataset["az"].apply(lambda deg: np.deg2rad(deg))
    if az:
        dataset["az"] = dataset["az"].apply(lambda rad: np.arctan2(np.sin(rad), np.cos(rad)))
    if log10_mc_energy:
        dataset["log10_mc_energy"] = dataset["mc_energy"].apply(lambda energy: np.log10(energy))
    if hdf5_file:
        dataset["hdf5_filepath"] = dataset[["folder", "source"]].apply(lambda x: path.join(x.folder, x.source), axis=1)#sintaxis para python 3.13
    return dataset

def filter_dataset(dataset, version, telescopes=[], number_of_observations=[], domain={}):
    """
    Select a subset from the dataset given some restrictions.

    The dataset can be filtered by telescope types, number of observations,
    and by a range of values for the targets. 

    Parameters
    ==========
    telescopes : `list` of `str` or 'str'
        Selected telescopes type for the dataset. 'str' if is just one.
    number_of_observations : `list` of `int` or 'int
        For each telescope in 'telescopes' parameter, the minimum amount
        of observations. 'int' if is just one.
    domain : `dict` [ `str`,  `tuple` of `int`]
        A dictionary with names of columns and their value range.
    Returns
    =======
     `pd.DataFrame`
        Filtered dataset.
    """
    if isinstance(telescopes, str):
        telescopes = [TELESCOPES_ALIAS[version][telescopes]]
    if isinstance(number_of_observations, int):
        number_of_observations = [number_of_observations]

    # filter telescopes
    filtered_dataset = dataset[dataset.type.isin(telescopes)]
    # # filter number of observatoins
    # FIXME: 
    # for telescope, observations in zip(telescopes, number_of_observations):
    #     filtered_events = filtered_dataset[filtered_dataset.type == telescope]\
    #                         .groupby("event_unique_id")\
    #                         .filter(lambda g: len(g) >= observations)\
    #                         .event_unique_id.unique()
    #     filtered_dataset = filtered_dataset[filtered_dataset.event_unique_id.isin(filtered_events)]
    # filter domain
    selection = np.ones((len(filtered_dataset),), dtype=bool)
    #if version == "DL1":#DL1 se encuentra en grados en vez de radianes
    #    for target, (vmin, vmax) in domain.items():
    #        if target == "alt":
    #            selection &= ((vmin*360/(2*np.pi) <= filtered_dataset[target]) & (filtered_dataset[target] <= vmax*360/(2*np.pi)))
    #        elif target == "az":
    #            angulo = filtered_dataset[target]*2*np.pi/360
    #            for i, a in enumerate(angulo):
    #                if a > np.pi:
    #                    angulo[i] -= np.pi
    #            selection &= ((vmin <= angulo) & (angulo <= vmax))
    #else:
    for target, (vmin, vmax) in domain.items():
        selection &= ((vmin <= filtered_dataset[target]) & (filtered_dataset[target] <= vmax))
    return filtered_dataset[selection]

def __get_resolution(targets, targets_domain, targets_shape):
    """Return the targets resolution for each target given the targets shape"""
    targets_resolution = {}
    for target in targets:
        vmin, vmax = targets_domain[target]
        shape = targets_shape[target]
        targets_resolution[target]  = (vmax -vmin) / shape
    return targets_resolution

def get_target_mode_config():
    targets = ["alt", "az"]
    target_shapes = {
        "alt": 81, 
        "az": 81, 
        "log10_mc_energy": 81
    }
    target_domains = {
        "alt": [1.15, 1.3], 
        "az": [-0.25, 0.25], 
        "log10_mc_energy": [-2.351, 2.47]
    }
    target_mode = "one_cell"
    if  target_shapes is not None:
        target_resolutions = __get_resolution(targets, target_domains, target_shapes)
        target_mode_config = {
            "target_shapes":      tuple([target_shapes[target]      for target in targets]),
            "target_domains":     tuple([target_domains[target]     for target in targets]),
            "target_resolutions": tuple([target_resolutions[target] for target in targets])
        }
    else:
        target_mode_config = {
            "target_domains":     tuple([target_domains[target]     for target in targets]),
            "target_shapes":      tuple([np.inf      for target in targets]),
            "target_resolutions": tuple([np.inf      for target in targets])
        }
    return target_mode_config

<>:41: SyntaxWarning: invalid escape sequence '\p'
<>:41: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_16518/370453913.py:41: SyntaxWarning: invalid escape sequence '\p'
  Translate domain from [0, 2\pi] to [-\pi, \pi]. (default=False)


Funciones de data.misc.py para el alineamiento de pixeles de las cámaras

In [3]:
def LST_LSTCam_align(pixels_position_array):
    xs = pixels_position_array[0]
    ys = pixels_position_array[1]
    # Distance matrix:
    delta_x = np.array([xs])-np.array([xs]).T
    delta_y = np.array([ys])-np.array([ys]).T
    dists = (delta_x**2+delta_y**2)**0.5
    angles = np.arctan2(delta_y, delta_x) # Angles from -pi to pi
    # Binary search, find maximum radious where no cell has more than 6 neighbors
    rad1 = 0
    rad2 = np.max(dists)
    for i in range(1000):
        rad = (rad1+rad2)/2.0
        neighs = dists<rad # matrix with true if i,j are neighbors
        np.fill_diagonal(neighs,False)
        max_neighs = np.max(np.sum(neighs,axis=1))
        if max_neighs>6:
            rad2 = rad
        else:
            rad1 = rad
    #
    rad = rad1
    neighs = dists<rad
    # Get a group of angles on an interval:
    ang_start = 0
    ang_end = np.pi*(6//2)
    # Neighbors with angle between those two
    conditions = np.all([neighs,angles>=ang_start,angles<ang_end],axis=0)
    neighbors = np.where(conditions)
    neigh_angles = angles[neighbors]
    # From the angles in this group, pick the median as the main axis
    main_axis_ang = np.median(neigh_angles)
    main_x = np.cos(main_axis_ang)
    main_y = np.sin(main_axis_ang)
    # Apply transformation
    tx = xs*main_x+ys*main_y
    ty = xs*main_y-ys*main_x
    # Now compute the maximum separation between neighboors in the main axis.
    dx = np.max(delta_x[neighs]*main_x+delta_y[neighs]*main_y)
    # Scale main axis by half of that separation:
    tx = np.round(tx/(dx/2.0))
    # Now compute the maximum separation between neighboors in the secondary axis.
    dy = np.max(delta_x[neighs]*main_y-delta_y[neighs]*main_x)
    # Scale secondary axis by that separation:
    ty = np.round(ty/dy)
    return np.stack((tx, ty))

def to_simple_and_shift(pixels_position_array):
    # get pixels positions
    xs = pixels_position_array[0]
    ys = pixels_position_array[1]
    # indices of x and y pixels position
    i = np.arange(0, len(ys))
    # row values of the telescope
    y_levels = np.sort(np.unique(ys))
    # image dimension
    nrows = len(y_levels)
    ncols = len(np.unique(xs))//2 + 1
    # new translated pixel positions
    new_x_l = np.copy(xs) # new pixels x positions left shift
    new_x_r = np.copy(xs) # new pixels x positions right shift
    new_y = np.copy(ys)
    # shift odd rows
    dx = 0
    for level, y_value in enumerate(y_levels):
        indices = i[ys == y_value]
        if dx == 0:
            dx = np.diff(np.sort(xs[indices])).min()/2
        if level % 2 != 0:
            new_x_l[indices] -= dx
            new_x_r[indices] += dx
    # round values
    new_x_l = np.round(new_x_l, 3)
    new_x_r = np.round(new_x_r, 3)
    # max indices of image output
    max_col_l = len(np.unique(new_x_l)) - 1
    max_col_r = len(np.unique(new_x_r)) - 1
    max_row = nrows - 1
    # apply linear transfomation
    new_x_l = ((max_col_l/(new_x_l.max() - new_x_l.min())) * (new_x_l - new_x_l.min()))
    new_x_l = np.round(new_x_l).astype(int)
    new_x_r = ((max_col_r/(new_x_r.max() - new_x_r.min())) * (new_x_r - new_x_r.min()))
    new_x_r = np.round(new_x_r).astype(int)
    new_y = ((max_row/(new_y.max() - new_y.min())) * (new_y - new_y.min()))
    new_y = np.round(new_y).astype(int)
    # prepare output
    simple = np.vstack((new_x_r, new_y))
    simple_shift = np.vstack((new_x_l, new_x_r, new_y))
    return simple, simple_shift

In [ ]:
file_path = "/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Dataset CTAO Monte Carlo Simulations - Eventlist on DL2 prod5/7298569/gamma-diffuse_with_images_00.dl2.h5"
output_path = "/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Dataset CTAO Monte Carlo Simulations - Eventlist on DL2 prod5/test CSV"
train_events_path = "/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/dataset/train_events.csv"
train_telescopes_path = "/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/dataset/train_telescopes.csv"
mode = "w"
version = "DL1"
pix_pos_folder = "/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions"

In [6]:
ed, td = extract_data(file_path,"DL1")
print("Número de eventos: " + str(len(ed)))
print("Número de telescopios activados: " + str(len(td)))

Número de eventos: 33922
Número de telescopios activados: 92803


Última sección de la función generate dataset en data/dataset.py en donde se guardan los archivos csv a partir de la información obtenida

In [18]:
events_filepath = path.join(output_path, "events.csv")
telescope_filepath = path.join(output_path, "telescopes.csv")
events_info_csv = open(events_filepath, mode=mode)
telescope_info_csv = open(telescope_filepath, mode=mode)
telescope_writer = csv.DictWriter(telescope_info_csv, delimiter=";", fieldnames=_telescope_fieldnames, lineterminator="\n")
events_writer = csv.DictWriter(events_info_csv, delimiter=';', fieldnames=_event_fieldnames, lineterminator="\n")
events_writer.writeheader()
telescope_writer.writeheader()
try:
    events_writer.writerows(ed)
    telescope_writer.writerows(td)
except:
    print("Error guardando datos.")
telescope_info_csv.close()
events_info_csv.close()

In [13]:
print(td[0])

{'telescope_id': 1, 'event_unique_id': 'a37133856e054ff99748', 'type': 'LST', 'x': -70.93000030517578, 'y': -52.06999969482422, 'z': 43.0, 'observation_indice': 5712}


In [3]:
a = [True,False,False,True]
b = np.array([1,2,3,4])
mask = b > 2
print(b[a])
for i,c in enumerate(a):
    print(str(i) + " " + str(c))

[1 4]
0 True
1 False
2 False
3 True


In [9]:
event_unique_id = uuid.uuid4().hex[:20]
print(type(event_unique_id))
a = 3
print(f'tel_{a:03d}')

<class 'str'>
tel_003


Simulación de los pasos para preparar y cargar los datos de entrenamiento en train/train_model.py contiene referencias a funciones de data/dataset.py y cdata/camera.py 

In [11]:
target_mode_config = get_target_mode_config()
target_domains_list = target_mode_config["target_domains"]

target_domains = {target: target_domain for target, target_domain in zip(["alt", "az", "log10_mc_energy"], target_domains_list)}

dataset = load_dataset(train_events_path, train_telescopes_path, None)
dataset = aggregate_dataset(dataset, version, az=True, log10_mc_energy=True)
dataset = filter_dataset(dataset,version, telescopes ="MST_FlashCam", number_of_observations = 3, domain = target_domains)

hdf5_file = tables.open_file(file_path, "r")
telescopes = dataset["type"].unique()
    # build list with loaded images
respond = [None] * len(dataset)
indices = np.arange(len(dataset))
    # iterate over file
            # and over telescope tables
for telescope_type in telescopes:
    #telescope_alias = TELESCOPES_ALIAS[version][telescope_type]
                # select indices for this file and telescope
    selector = (dataset["hdf5_filepath"] == file_path) & (dataset["type"] == telescope_type)
    observations_indices_selected = dataset[selector]["observation_indice"]#.to_numpy()
    observations_indices_selected = np.array([list(map(int, row.split("_"))) for row in observations_indices_selected])
    respond_indices_selected = indices[selector]
                # load images and copy results
    #images = hdf5_file.root.dl1.event.telescope.images[telescope_alias][observations_indices_selected]
    #for i, img in zip(respond_indices_selected, images):
        #respond[i] = (img[_images_attributes[version]["charge"]], img[_images_attributes[version]["peakpos"]]) 
    for i, obs_ind_data in zip(respond_indices_selected,observations_indices_selected):
        obs_id = obs_ind_data[0]
        event_id = obs_ind_data[1]
        tel = obs_ind_data[2]
        #file = obs_ind_data[3]
        tel_name = f'tel_{tel:03d}'
        tel_tabla = hdf5_file.root.dl1.event.telescope.images[tel_name]
        #img = [x for x in tel_tabla.iterrows() if x["obs_id"] == obs_id and x["event_id"] == event_id]
        #img = img[0]
        for img in tel_tabla:
            if img["obs_id"] == obs_id and img["event_id"] == event_id:
                break
        #df_img = pd.DataFrame(tel_tabla.read())#esta parte no funciona
        #mask = (
        #    df_img["event_id"].isin(event_id) &
        #    df_img["obs_id"].isin(obs_id)
        #)

        #item = df_img[mask]
        respond[i] = (img[_images_attributes[version]["charge"]], img[_images_attributes[version]["peakpos"]])

KeyboardInterrupt: 

Función para crear los archivos de pixels_positions ubicada en data/misc.py

In [5]:
hdf5_file = tables.open_file(file_path, "r")
inverse_alias = {TELESCOPES_ALIAS[version][t]:t for t in TELESCOPES}
modes = ('raw', 'simple', 'simple-shift')
pixpos_folder = join(pix_pos_folder, version)
print(pixpos_folder)
type_geometry_dict = {}
raw_pixpos = {}
all_pixpos = {}
telescopes_info = hdf5_file.root.configuration.instrument.subarray.layout
geometry_group = hdf5_file.root.configuration.instrument.telescope.camera
for tel in telescopes_info:
    tipo = tel["type"].decode("utf-8")
    if tipo not in type_geometry_dict:
        type_geometry_dict[tipo] = tel["camera_index"]
for t in type_geometry_dict:
    if t not in inverse_alias:
        continue
    geometry_index = "geometry_" + str(type_geometry_dict[t])
    raw_pixpos[t] = []
    geometry = geometry_group[geometry_index]
    num_pixels = len(geometry)
    #for pixel in geometry:
        #raw_pixpos[t].append([pixel["pix_x"],pixel["pix_y"]])
    raw_pixpos[t].append(geometry.cols.pix_x[:])
    raw_pixpos[t].append(geometry.cols.pix_y[:])
all_pixpos['raw'] = {}
for telescope, pixpos in raw_pixpos.items():
    if telescope == TELESCOPES_ALIAS[version]["LST_LSTCam"]:
        LST_LSTCam_not_aligm = pixpos
        all_pixpos['raw']['LST_LSTCam_not_aligm'] = LST_LSTCam_not_aligm
        raw_pixpos[telescope] = LST_LSTCam_align(pixpos)
        pixpos = raw_pixpos[telescope]
        np.savetxt(join(pixpos_folder,'raw', f'{telescope}_not_align.npy'), LST_LSTCam_not_aligm)
    if telescope == TELESCOPES_ALIAS[version]["MST_FlashCam"]:
        MST_FlashCam_not_aligm = pixpos
        all_pixpos['raw']['MST_FlashCam_not_aligm'] = MST_FlashCam_not_aligm
        raw_pixpos[telescope] = LST_LSTCam_align(pixpos)
        pixpos = raw_pixpos[telescope]
        np.savetxt(join(pixpos_folder,'raw', f'{telescope}_not_align.npy'), MST_FlashCam_not_aligm)
    np.savetxt(join(pixpos_folder,'raw', f'{telescope}.npy'), pixpos)
    all_pixpos['raw'][telescope] = pixpos
all_pixpos['simple'] = {}
all_pixpos['simple_shift'] = {}
all_pixpos["time"] = {}
all_pixpos["time_shift"] = {}
for telescope, pixpos in raw_pixpos.items():
    simple, shift = to_simple_and_shift(pixpos)
    all_pixpos['simple'][telescope] = simple
    all_pixpos['time'][telescope] = simple
    all_pixpos['simple_shift'][telescope] = shift
    all_pixpos['time_shift'][telescope] = shift
    np.savetxt(join(pixpos_folder,'simple', f'{telescope}.npy'), simple)
    np.savetxt(join(pixpos_folder,'time', f'{telescope}.npy'), simple)
    np.savetxt(join(pixpos_folder,'simple_shift', f'{telescope}.npy'), shift)
    np.savetxt(join(pixpos_folder,'time_shift', f'{telescope}.npy'), shift)

/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/DL1


In [53]:
load_data = np.load('/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/ML1/raw/LST_LSTCam_not_align.npy')

In [56]:
print((load_data[0]))

[ 0.         -0.00944877 -0.0472442  ... -0.6519913  -0.6141959
 -0.62364465]


In [6]:
TELESCOPE_CAMERA   = {
    "LST_LSTCam":    "LSTCam", 
    "MST_FlashCam":  "FlashCam", 
    "SST1M_DigiCam": "DigiCam"
}

pixpos_folder = "/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions"
PIXELS_POSITION = {}

for version in ("ML1", "ML2", "DL1"):
    PIXELS_POSITION[version] = {}
    for mode in ("raw", "simple", "simple_shift", "time", "time_shift"):
        PIXELS_POSITION[version][mode] = {}
        for telescope in TELESCOPES:
            path_  = path.join(pixpos_folder, version, mode, f"{telescope}.npy")
            try:
                pixpos = np.loadtxt(path_, dtype=float)
                pixpos = pixpos.astype(int) if  mode != 'raw' else pixpos
                PIXELS_POSITION[version][mode][telescope] = pixpos
            except OSError as err:
                print(err)
                print("File %s/%s/%s not found" %(version, mode, telescope))
#except OSError as err:
#    print(err)
#    print('Try running extract_pixel_positions to generate pixpos files.')

__scalers_folder = "/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/scalers"

__default_scalers = {
    "ML1_array-scaler": join(__scalers_folder, "ML1_array-scaler.gz"),
    "ML1_LST_LSTCam_peak_scaler" : join(__scalers_folder, "ML1_LST_LSTCam_peak_scaler.gz"),
    "ML1_MST_FlashCam_peak_scaler" : join(__scalers_folder, "ML1_MST_FlashCam_peak_scaler.gz"),
    "ML1_SST1M_DigiCam_peak_scaler" : join(__scalers_folder, "ML1_SST1M_DigiCam_peak_scaler.gz")
}

/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/DL1/raw/SST1M_DigiCam.npy not found.
File DL1/raw/SST1M_DigiCam not found
/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/DL1/simple/SST1M_DigiCam.npy not found.
File DL1/simple/SST1M_DigiCam not found
/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/DL1/simple_shift/SST1M_DigiCam.npy not found.
File DL1/simple_shift/SST1M_DigiCam not found
/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/DL1/time/SST1M_DigiCam.npy not found.
File DL1/time/SST1M_DigiCam not found
/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/DL1/time_shift/SST1M_DigiCam.npy not found.
File DL1/time_shift/SST1M_DigiCam not found


In [23]:
def load_camera_geometry(telescope_type, version="ML1"):
    geometry = CameraGeometry.from_name(TELESCOPE_CAMERA[telescope_type])
    #alias = TELESCOPES_ALIAS[version][telescope_type]
    pixpos = PIXELS_POSITION[version]['raw'][telescope_type]
    geometry.pix_x = units.quantity.Quantity(pixpos[0], 'meter')
    geometry.pix_y = units.quantity.Quantity(pixpos[1], 'meter')
    return geometry

def load_scaler(default_name_or_custom_scaler_path):
    if default_name_or_custom_scaler_path in __default_scalers:
        return load(__default_scalers[default_name_or_custom_scaler_path])
    elif default_name_or_custom_scaler_path is None:
        return None
    elif exists(default_name_or_custom_scaler_path):
        return load(default_name_or_custom_scaler_path)
    else:
        raise OSError(f"Scaler not found: {default_name_or_custom_scaler_path}")

class CameraPipe():
    def __init__(self, telescope_type, charge_scaler_value=None, peak_scaler_path=None, tailcuts_clean_params=None, version="ML1"):
        self.telescope_type = telescope_type
        self.charge_scaler_value = charge_scaler_value if charge_scaler_value is not None else 1
        self.peak_scaler = load_scaler(peak_scaler_path)
        self.tailcuts_clean_params = tailcuts_clean_params #{picture_thresh:int, boundary_thresh:int}
        self.geometry = load_camera_geometry(telescope_type, version)
        self.version = version

    def __call__(self, cameras):
        if not isinstance(cameras, list):
            cameras = [cameras]
        results = []
        for (charge, peak) in cameras:
            if self.tailcuts_clean_params is not None:
                cleanmask = tailcuts_clean(self.geometry, charge, **self.tailcuts_clean_params)
                charge[~cleanmask] = 0.0
                for _ in range(3):
                    cleanmask = dilate(self.geometry, cleanmask)
                peak[~cleanmask] = 0.0
            charge /= self.charge_scaler_value
            if self.peak_scaler is not None:
                peak  = self.peak_scaler.transform(peak.reshape((-1, 1))).flatten()
            results.append((charge, peak))
        return results
    
class TelescopeFeaturesPipe():
    def __init__(self, telescope_type=None, array_scaler_path=None, version="ML1"):
        self.telescope_type = telescope_type
        self.array_scaler = load_scaler(array_scaler_path)
        self.version = version

    def __call__(self, telescope_features):
        if self.array_scaler is not None: 
            if len(telescope_features.shape) == 1:
                telescope_features = telescope_features.reshape((1, -1))
            return self.array_scaler.transform(telescope_features)

In [26]:
params = {
            "charge_scaler_value" : 62.652400970459,
            "peak_scaler_path" : "ML1_MST_FlashCam_peak_scaler",
            "tailcuts_clean_params": {"boundary_thresh": 6, "picture_thresh": 14}
        }
camera_pipe = CameraPipe(telescope_type="MST_FlashCam",version="DL1",**params)

tel_f_pipe = {
    "array_scaler_path": "ML1_array-scaler"
}
telescope_f = TelescopeFeaturesPipe(telescope_type="MST_FlashCam", version="DL1", **tel_f_pipe)

/home/sven-klein-plarre/anaconda3/envs/gerumo_py313/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 0.23.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/sven-klein-plarre/anaconda3/envs/gerumo_py313/lib/python3.13/site-packages/ctapipe/instrument/camera/geometry.py:604: FromNameWarning: .from_name uses pre-defined data that is likely different from the data being analyzed. Access instrument information via the SubarrayDescription instead.
  warn_from_name()
/home/sven-klein-plarre/anaconda3/envs/gerumo_py313/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 0.23.1 when using version 1.8.0. This might lead to breaking code o

In [28]:
print(type(camera_pipe))
print(type(telescope_f))

<class '__main__.CameraPipe'>
<class '__main__.TelescopeFeaturesPipe'>


Generación de Alfas para el telescopio MST y LST DL1 usando el dataset completo asumiendo datos de entrada de ambos csv

In [8]:
def estimate_alphas(dataset, column="log10_mc_energy", bins=81, rescale=None):
    """
    Estimate alpha values for focal loss or weighted loss functions.

    Parameters
    ==========
    dataset :  `pd.DataFrame`
        Loaded dataset. Dataset of the events CSV with mc_energy transformed to log10
    column : `str`
        Target name, dataset column.
    bins : `int`
        Number of bins or classes.
    rescale : (`int`, `int`) or `None`
        Rescale value between given range.
    Returns
    -------
        `np.ndarray`
            Bins' weights (alpha values).
    """
    count, bins = np.histogram(dataset[column], bins=bins)
    count = count / (count.sum())
    alphas = (1 - count)
    if rescale is not None:
        min_, max_ = rescale
        alphas = (alphas - alphas.min()) * (max_ - min_) / (alphas.max() - alphas.min()) + min_
    else:
        alphas /= alphas.sum()
    return alphas

In [29]:
target_mode_config = get_target_mode_config()
target_domains_list = target_mode_config["target_domains"]
events_path = "/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/dataset/dataset completo/events.csv"
telescopes_path = "/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/dataset/dataset completo/telescopes.csv"

target_domains = {target: target_domain for target, target_domain in zip(["alt", "az", "log10_mc_energy"], target_domains_list)}

dataset = load_dataset(events_path, telescopes_path, None)
dataset = aggregate_dataset(dataset, version, az=True, log10_mc_energy=True)
dataset_MST = filter_dataset(dataset,version, telescopes ="MST_FlashCam", number_of_observations = 3, domain = target_domains)
dataset_LST = filter_dataset(dataset,version, telescopes ="LST_LSTCam", number_of_observations = 3, domain = target_domains)

alf_MST = estimate_alphas(dataset_MST, rescale=(0.1,1))
alf_LST = estimate_alphas(dataset_LST, rescale=(0.1,1))
print(alf_MST)
print(alf_LST)

[0.99999506 1.         0.99997038 0.99990128 0.99984698 0.99965448
 0.99928428 0.9984106  0.99733455 0.99441736 0.98911113 0.97966358
 0.96351785 0.93673498 0.89874241 0.84246187 0.77139794 0.68445975
 0.58732868 0.49292723 0.406532   0.33290628 0.27234127 0.21821283
 0.18963325 0.16311693 0.139271   0.12349547 0.10933402 0.1
 0.10161902 0.11152562 0.11705396 0.14202037 0.16029353 0.19898208
 0.2243878  0.25670888 0.30321609 0.35421015 0.3925828  0.43340865
 0.47326211 0.51376712 0.54894122 0.58348845 0.61294664 0.64687687
 0.68256926 0.70142487 0.73144576 0.76105203 0.77250361 0.79293381
 0.82017079 0.83034393 0.85140101 0.8702813  0.87884036 0.89234039
 0.90352048 0.90782963 0.91464134 0.93184832 0.93096477 0.93752968
 0.94987468 0.95305348 0.96157306 0.96222955 0.96913998 0.96992974
 0.97392792 0.97543341 0.98235865 0.98415043 0.9825709  0.99130766
 0.99208262 0.99309944 0.99375593]
[1.         0.99895556 0.99754335 0.99485134 0.98954087 0.98106765
 0.96791651 0.94623331 0.9097661  

In [30]:
def get_alphas(telescope, version):
    "Alpha values for focal loss precomputed for log10 mc_energy with 81 bins in range (0.1, 1)"
    return {"ML1": {
        'SST1M_DigiCam': np.array(
               [0.99981382, 1.        , 1.        , 1.        , 0.99981382,
                0.99962764, 0.99832437, 0.99795201, 0.99664874, 0.99292511,
                0.9910633 , 0.98305751, 0.97300372, 0.94898635, 0.91696318,
                0.87190732, 0.79948283, 0.71030203, 0.61702524, 0.50717832,
                0.36623914, 0.30554406, 0.23758792, 0.22809268, 0.21245345,
                0.12736864, 0.11210178, 0.13425734, 0.14635912, 0.1       ,
                0.155482  , 0.24205627, 0.16423252, 0.17912702, 0.18043029,
                0.26030203, 0.33998759, 0.32658254, 0.3546959 , 0.32341746,
                0.35190319, 0.42525859, 0.50885395, 0.37964419, 0.55111709,
                0.48893256, 0.59580058, 0.5747621 , 0.63117501, 0.50680596,
                0.68386429, 0.67772031, 0.69950352, 0.7125362 , 0.6749276 ,
                0.66133637, 0.72612743, 0.75740588, 0.87991312, 0.73953248,
                0.78831196, 0.84155978, 0.83727762, 0.86576334, 0.87470004,
                0.84453868, 0.87637567, 0.89182871, 0.75666115, 0.86613571,
                0.92813405, 0.94582127, 0.95624741, 0.93576748, 0.9731899 ,
                0.96685974, 0.95810923, 0.94377327, 0.96779065, 0.95326851,
                0.98473314]),
        'MST_FlashCam': np.array(
               [1.        , 1.        , 0.99969724, 0.99969724, 0.99798161,
                0.99646782, 0.99576138, 0.98980713, 0.98143081, 0.96447634,
                0.93026463, 0.89554833, 0.83388652, 0.71782911, 0.62246019,
                0.47340211, 0.3558309 , 0.2185804 , 0.18608432, 0.10787172,
                0.10262391, 0.11745907, 0.10655977, 0.115037  , 0.1       ,
                0.10131195, 0.14672572, 0.12078941, 0.11231218, 0.16034985,
                0.24058085, 0.26742543, 0.27408612, 0.29982059, 0.36329895,
                0.38953801, 0.43131868, 0.44978695, 0.5142745 , 0.47400763,
                0.52366001, 0.58633102, 0.66706661, 0.61226732, 0.66111236,
                0.6804889 , 0.70753532, 0.75849966, 0.73145324, 0.7912985 ,
                0.76879345, 0.83681319, 0.82510653, 0.78655528, 0.84831801,
                0.87122673, 0.87082305, 0.88495178, 0.85053824, 0.87919937,
                0.91704418, 0.95670554, 0.9217874 , 0.93359498, 0.93026463,
                0.9565037 , 0.95034761, 0.95640278, 0.96750392, 0.93773268,
                0.92108096, 0.96074232, 0.98768782, 0.9880915 , 0.98052254,
                0.99515586, 0.98768782, 0.98435748, 0.98223817, 0.98455932,
                0.99555954]),
        'LST_LSTCam': np.array(
               [0.99395071, 0.98924571, 0.98386856, 0.95899925, 0.93244959,
                0.88640777, 0.83162808, 0.74861837, 0.64846901, 0.56445108,
                0.4942121 , 0.38599701, 0.34600448, 0.25056012, 0.22266617,
                0.2401419 , 0.12352502, 0.1       , 0.14301718, 0.21863331,
                0.19746079, 0.20888723, 0.27744586, 0.28752801, 0.34432412,
                0.37793129, 0.36616878, 0.37490665, 0.43338312, 0.50093353,
                0.48480209, 0.52076176, 0.54663928, 0.61620612, 0.60746826,
                0.6165422 , 0.65989544, 0.70056012, 0.69047797, 0.7529873 ,
                0.75097087, 0.75399552, 0.75466766, 0.78961912, 0.82389843,
                0.82961165, 0.84473488, 0.86456311, 0.8729649 , 0.89111277,
                0.89850635, 0.92270351, 0.90354742, 0.93547423, 0.93043316,
                0.91430172, 0.93749066, 0.94522031, 0.95026139, 0.95698282,
                0.94085138, 0.96504854, 0.97815534, 0.97647498, 0.97781927,
                0.98017177, 0.98487677, 0.9821882 , 0.98857356, 0.98924571,
                0.98151606, 0.96840926, 0.99058999, 0.99395071, 0.9976475 ,
                0.99428678, 1.        , 0.99932786, 0.99831964, 0.995295  ,
                0.99395071])
        },
        "DL1" : {
            'MST_FlashCam' : np.array(
                [0.99999506, 1.     ,    0.99997038, 0.99990128, 0.99984698, 0.99965448,
                0.99928428, 0.9984106,  0.99733455, 0.99441736, 0.98911113, 0.97966358,
                0.96351785, 0.93673498, 0.89874241, 0.84246187, 0.77139794, 0.68445975,
                0.58732868, 0.49292723, 0.406532,   0.33290628, 0.27234127, 0.21821283,
                0.18963325, 0.16311693, 0.139271,   0.12349547, 0.10933402, 0.1,
                0.10161902, 0.11152562, 0.11705396, 0.14202037, 0.16029353, 0.19898208,
                0.2243878,  0.25670888, 0.30321609, 0.35421015, 0.3925828,  0.43340865,
                0.47326211, 0.51376712, 0.54894122, 0.58348845, 0.61294664, 0.64687687,
                0.68256926, 0.70142487, 0.73144576, 0.76105203, 0.77250361, 0.79293381,
                0.82017079, 0.83034393, 0.85140101, 0.8702813,  0.87884036, 0.89234039,
                0.90352048, 0.90782963, 0.91464134, 0.93184832, 0.93096477, 0.93752968,
                0.94987468, 0.95305348, 0.96157306, 0.96222955, 0.96913998, 0.96992974,
                0.97392792, 0.97543341, 0.98235865, 0.98415043, 0.9825709,  0.99130766,
                0.99208262, 0.99309944, 0.99375593]),
            'LST_LSTCam' : np.array(
                [1.        , 0.99895556, 0.99754335, 0.99485134, 0.98954087, 0.98106765,
                0.96791651, 0.94623331, 0.9097661 , 0.86676746, 0.80058515, 0.73221098,
                0.64865563, 0.57348523, 0.49269544, 0.43751655, 0.37383501, 0.33585263,
                0.29923833, 0.26712541, 0.23362972, 0.20384106, 0.19345548, 0.15488469,
                0.14839738, 0.14451382, 0.11757899, 0.10832611, 0.10607542, 0.12087413,
                0.1       , 0.14099802, 0.15637044, 0.18252562, 0.21150521, 0.25532763,
                0.26972916, 0.29682581, 0.34457593, 0.38981056, 0.43241202, 0.46392181,
                0.49781468, 0.54487341, 0.57719227, 0.60177343, 0.63540151, 0.66314542,
                0.70117193, 0.72122227, 0.74346447, 0.77597457, 0.78534512, 0.80501299,
                0.82532812, 0.84202448, 0.8616188 , 0.88184567, 0.89073078, 0.90520586,
                0.91622399, 0.92109315, 0.92265246, 0.9389075 , 0.94159952, 0.94857227,
                0.95838414, 0.96097318, 0.96681323, 0.96862261, 0.97513934, 0.97559536,
                0.97822854, 0.98042039, 0.98558376, 0.98839346, 0.98624573, 0.99333617,
                0.99357153, 0.99495432, 0.99545447])
        }}[version][telescope]
print(get_alphas("LST_LSTCam","DL1"))

[1.         0.99895556 0.99754335 0.99485134 0.98954087 0.98106765
 0.96791651 0.94623331 0.9097661  0.86676746 0.80058515 0.73221098
 0.64865563 0.57348523 0.49269544 0.43751655 0.37383501 0.33585263
 0.29923833 0.26712541 0.23362972 0.20384106 0.19345548 0.15488469
 0.14839738 0.14451382 0.11757899 0.10832611 0.10607542 0.12087413
 0.1        0.14099802 0.15637044 0.18252562 0.21150521 0.25532763
 0.26972916 0.29682581 0.34457593 0.38981056 0.43241202 0.46392181
 0.49781468 0.54487341 0.57719227 0.60177343 0.63540151 0.66314542
 0.70117193 0.72122227 0.74346447 0.77597457 0.78534512 0.80501299
 0.82532812 0.84202448 0.8616188  0.88184567 0.89073078 0.90520586
 0.91622399 0.92109315 0.92265246 0.9389075  0.94159952 0.94857227
 0.95838414 0.96097318 0.96681323 0.96862261 0.97513934 0.97559536
 0.97822854 0.98042039 0.98558376 0.98839346 0.98624573 0.99333617
 0.99357153 0.99495432 0.99545447]
